# 5 - The Base Taxonomy: *what* is asked, *what* is true

A **base** entry defines a question type mined from a GMS store, whose ground truth is computed by a GMS primitive (and is therefore provably correct). Base entries are domain-agnostic templates; pick the families and categories your domain supports.

This complements the **enrichment** catalog (notebook 2), which only changes *how* a question is presented, never the answer.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "code"))   # import gmstest from the tutorial root
import gmstest as gt
cat = gt.Catalog.load()
print(cat.summary())

## The five families
The 18 base categories are the deduplicated union of the JASA-10, the 13-category testing taxonomy, and the 6 knowlytix generators.

In [ ]:
for fam, names in cat.families.items():
    print(f'{fam:24s} {len(names):2d}  {", ".join(names)}')

## One entry in detail
Each base carries the graph capability it `requires`, the `generator` that mines it (or `None` if not yet built), its `answer_type`, the GMS primitive that defines `ground_truth`, and a `status`.

In [ ]:
b = cat.bases['exact_recall']
for f in ('family','requires','generator','answer_type','ground_truth','status','source'):
    print(f'{f:14s} {getattr(b, f)}')

## Three ways to supply base questions
**`UserBaseSource`** - bring your own `(query, answer)` pairs. No GMS is used: your answer *is* the ground truth, and queries are not expanded by a store. This is the lightweight path for augmenting an existing labeled set.

In [ ]:
user = gt.UserBaseSource([
    {'query': 'What is the overdraft fee?', 'answer': '35'},
    {'query': 'How long to dispute a charge?', 'answer': '60 days'},
])
items = user.items()
for it in items:
    print(it.qid, '|', it.query, '->', it.answer, '| base =', it.base)

**`SeedCaseSource`** - hand-labeled cases carrying *per-component* ground truth (classification, product, issue, policy, escalation). This is what trajectory + weak-link testing of an agent consumes.

In [ ]:
cases = [{'id': 'case-001', 'message': 'I was charged a $35 overdraft fee and want it removed.',
          'expected_classification': 'complaint', 'expected_escalation': False,
          'expected_product': 'checking_account', 'expected_issue': 'overdraft_fee'}]
it = gt.SeedCaseSource(cases).items()[0]
print('qid       :', it.qid)
print('query     :', it.query)
print('components:', it.components)

**`CatalogBaseSource`** - mine a trained GMS store with the selected categories' generators (the provably-correct path). We run it on the real policy store; each mined question carries a graph-derived ground truth.

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

def _find_store(name="gms_policy_store_geode"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        c = p / "beyond-prompt-and-pray" / "code" / "data" / name
        if c.exists():
            return c
    raise FileNotFoundError(name + " not found; run scripts/build_geode_rag_store.py")

STORE = _find_store()
store = GMSExpertStore(DocGMSConfig(store_path=str(STORE), ingest_mode="regex"),
                       device=torch.device("cpu"))
assert store.load(), "store failed to load"
print("loaded store:", len(store.adapter.relation_to_idx), "relations,",
      len(store.adapter.entity_to_idx), "entities")

In [ ]:
suite_cat = gt.resolve(cat, ['exact_recall', 'counting', 'contradiction'], [], mode='cross')
cat_items = gt.CatalogBaseSource(store, max_per_category=3).items(suite_cat)
print(len(cat_items), 'questions mined from the store; examples:')
for it in cat_items[:3]:
    print(' ', it.base, '|', it.query[:56], '-> ground truth:', it.answer)

## Self-check

In [ ]:
assert len(cat.bases) == 18 and len(cat.families) == 5
assert cat.bases['exact_recall'].ground_truth == 'lookup_enm'
assert items[0].base is None              # user-supplied carries no catalog base
assert cat_items and cat_items[0].base in cat.bases   # mined from the real store
print('OK - base catalog')